<a href="https://colab.research.google.com/github/moshuhuang/RouteGuard-last-mile-address-risk/blob/main/01_data_preparation_and_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Preparation & EDA

This notebook loads raw public address data from OpenAddresses (Florida),
samples 30K records, injects controlled address errors based on real last-mile
delivery failure patterns, and produces the labeled dataset for modeling.

## Address Risk Taxonomy
Five error types were defined based on operational delivery failure patterns:

| Error Type | Description |
|---|---|
| `missing_unit_number` | Apartment or unit number missing for multi-unit locations |
| `wrong_or_incomplete_unit_or_house_number` | Unit/house number exists but is incomplete or invalid |
| `ambiguous_abbreviation` | Street or direction contains unclear abbreviations |
| `street_only` | Address contains only street-level info, no deliverable location |
| `missing_or_wrong_city_zip` | City or ZIP is missing, inconsistent, or mismatched |

## Dataset
**Output:** `address_risk_dataset_30k.csv`  
**Size:** 30,000 records | 80% valid / 20% risky  
**Source:** OpenAddresses — Florida county/city-level data (public dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
import pandas as pd

file_path = "/content/drive/MyDrive/Git Projects/source.geojson"

records = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        feature = json.loads(line)

        props = feature.get("properties", {})
        geometry = feature.get("geometry", {})
        coords = geometry.get("coordinates", [None, None])

        props["longitude"] = coords[0] if coords else None
        props["latitude"] = coords[1] if coords else None

        records.append(props)

df = pd.DataFrame(records)

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df["city"].value_counts().head(20)

In [ ]:
df_base = df.dropna(subset=["number", "street", "city", "postcode", "longitude", "latitude"]).copy()

df_base = df_base.sample(n=30000, random_state=42)

print(df_base.shape)
df_base.head()

In [ ]:
output_path = "/content/drive/MyDrive/Git Projects/address_base_sample_30k.csv"
df_base.to_csv(output_path, index=False)

print("Saved to:", output_path)

In [ ]:
import pandas as pd
import numpy as np
import re
import random

# Load the 30K valid address base
file_path = "/content/drive/MyDrive/Git Projects/address_base_sample_30k.csv"
df_base = pd.read_csv(file_path)

print(df_base.shape)
df_base.head()

In [ ]:
# Make a working copy
df = df_base.copy()

# Standardize key columns
for col in ["number", "street", "unit", "city", "postcode"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# Build original clean address
df["original_address"] = (
    df["number"] + " " +
    df["street"] + " " +
    df["unit"] + ", " +
    df["city"] + ", FL " +
    df["postcode"]
)

# Clean extra spaces and commas
df["original_address"] = (
    df["original_address"]
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(" ,", ",", regex=False)
    .str.strip()
)

df[["number", "street", "unit", "city", "postcode", "original_address"]].head()

In [ ]:
def clean_address_text(text):
    """Clean repeated spaces and awkward commas."""
    text = re.sub(r"\s+", " ", str(text))
    text = text.replace(" ,", ",")
    text = text.replace(",,", ",")
    return text.strip()


def make_valid_address(row):
    """Keep the original address as a valid input address."""
    return row["original_address"]


def inject_missing_unit_number(row):
    """Remove unit/apartment information if available; otherwise mimic a multi-unit address without unit."""
    number = row["number"]
    street = row["street"]
    city = row["city"]
    postcode = row["postcode"]

    # Remove unit field from the address
    address = f"{number} {street}, {city}, FL {postcode}"
    return clean_address_text(address)


def inject_wrong_or_incomplete_unit_or_house_number(row):
    """Create incomplete unit information or suspicious house number."""
    number = row["number"]
    street = row["street"]
    city = row["city"]
    postcode = row["postcode"]

    choice = np.random.choice(["incomplete_unit", "wrong_house_number"])

    if choice == "incomplete_unit":
        # Unit exists but is not specific enough
        unit_token = np.random.choice(["Apt", "Unit", "Ste", "#"])
        address = f"{number} {street} {unit_token}, {city}, FL {postcode}"
    else:
        # Shift house number to a suspicious value
        try:
            num_int = int(float(number))
            shifted_number = str(num_int + np.random.choice([300, 500, 700, 1000]))
        except:
            shifted_number = str(number) + "99"

        address = f"{shifted_number} {street}, {city}, FL {postcode}"

    return clean_address_text(address)


def inject_ambiguous_abbreviation(row):
    """Replace parts of street text with unclear abbreviations."""
    number = row["number"]
    street = row["street"]
    unit = row["unit"]
    city = row["city"]
    postcode = row["postcode"]

    street_new = str(street)

    replacements = {
        "WEST": "W",
        "EAST": "E",
        "NORTH": "N",
        "SOUTH": "S",
        "COURT": "CT",
        "STREET": "ST",
        "AVENUE": "AV",
        "ROAD": "RD",
        "DRIVE": "DR",
        "LANE": "LN",
        "PLACE": "PL",
        "CIRCLE": "CR",
        "TERRACE": "TER",
        "BOULEVARD": "BLV"
    }

    # Apply random replacements
    upper_street = street_new.upper()
    for full, abbr in replacements.items():
        if full in upper_street and np.random.rand() < 0.7:
            upper_street = upper_street.replace(full, abbr)

    # Sometimes make it more ambiguous by collapsing words into initials
    if np.random.rand() < 0.3:
        tokens = upper_street.split()
        if len(tokens) >= 2:
            upper_street = "".join([t[0] for t in tokens if len(t) > 0])

    address = f"{number} {upper_street} {unit}, {city}, FL {postcode}"
    return clean_address_text(address)


def inject_street_only(row):
    """Keep only street-level information without a specific deliverable location."""
    street = row["street"]
    city = row["city"]
    postcode = row["postcode"]

    variants = [
        f"{street}, {city}, FL {postcode}",
        f"{street}, {city}",
        f"{street}"
    ]

    return clean_address_text(np.random.choice(variants))


def inject_missing_or_wrong_city_zip(row, city_pool, zip_pool):
    """Remove or mismatch city/ZIP information."""
    number = row["number"]
    street = row["street"]
    unit = row["unit"]
    city = row["city"]
    postcode = row["postcode"]

    choice = np.random.choice(["missing_zip", "missing_city", "wrong_zip", "wrong_city"])

    if choice == "missing_zip":
        address = f"{number} {street} {unit}, {city}, FL"
    elif choice == "missing_city":
        address = f"{number} {street} {unit}, FL {postcode}"
    elif choice == "wrong_zip":
        wrong_zip = np.random.choice(zip_pool)
        while str(wrong_zip) == str(postcode) and len(zip_pool) > 1:
            wrong_zip = np.random.choice(zip_pool)
        address = f"{number} {street} {unit}, {city}, FL {wrong_zip}"
    else:
        wrong_city = np.random.choice(city_pool)
        while str(wrong_city).upper() == str(city).upper() and len(city_pool) > 1:
            wrong_city = np.random.choice(city_pool)
        address = f"{number} {street} {unit}, {wrong_city}, FL {postcode}"

    return clean_address_text(address)

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

df_risk = df.copy()

# Default: valid addresses
df_risk["input_address"] = df_risk["original_address"]
df_risk["error_type"] = "valid"
df_risk["is_address_risk"] = 0

# Define risky sample ratio
risk_ratio = 0.20
n_risky = int(len(df_risk) * risk_ratio)

# Select rows to inject address risks
risky_indices = np.random.choice(df_risk.index, size=n_risky, replace=False)

# Define error type distribution
error_types = [
    "missing_unit_number",
    "wrong_or_incomplete_unit_or_house_number",
    "ambiguous_abbreviation",
    "street_only",
    "missing_or_wrong_city_zip"
]

error_probs = [0.25, 0.25, 0.15, 0.20, 0.15]

assigned_errors = np.random.choice(
    error_types,
    size=n_risky,
    p=error_probs
)

city_pool = df_risk["city"].dropna().astype(str).unique()
zip_pool = df_risk["postcode"].dropna().astype(str).unique()

# Apply error injection
for idx, error_type in zip(risky_indices, assigned_errors):
    row = df_risk.loc[idx]

    if error_type == "missing_unit_number":
        new_address = inject_missing_unit_number(row)

    elif error_type == "wrong_or_incomplete_unit_or_house_number":
        new_address = inject_wrong_or_incomplete_unit_or_house_number(row)

    elif error_type == "ambiguous_abbreviation":
        new_address = inject_ambiguous_abbreviation(row)

    elif error_type == "street_only":
        new_address = inject_street_only(row)

    elif error_type == "missing_or_wrong_city_zip":
        new_address = inject_missing_or_wrong_city_zip(row, city_pool, zip_pool)

    df_risk.at[idx, "input_address"] = new_address
    df_risk.at[idx, "error_type"] = error_type
    df_risk.at[idx, "is_address_risk"] = 1

print(df_risk.shape)
df_risk[["original_address", "input_address", "error_type", "is_address_risk"]].head(10)

In [ ]:
print(df_risk["is_address_risk"].value_counts(normalize=True))
print()
print(df_risk["error_type"].value_counts())

In [ ]:
# Randomly inspect generated examples by error type
for error in [
    "missing_unit_number",
    "wrong_or_incomplete_unit_or_house_number",
    "ambiguous_abbreviation",
    "street_only",
    "missing_or_wrong_city_zip"
]:
    print("\n" + "="*80)
    print("ERROR TYPE:", error)
    display(
        df_risk[df_risk["error_type"] == error][
            ["original_address", "input_address", "error_type"]
        ].sample(8, random_state=42)
    )

In [ ]:
display(
    df_risk[df_risk["error_type"] == "valid"][
        ["original_address", "input_address", "error_type"]
    ].sample(10, random_state=42)
)

In [ ]:
output_path = "/content/drive/MyDrive/Git Projects/address_risk_dataset_30k.csv"

df_risk.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Final shape:", df_risk.shape)